In [1]:
import pandas as pd
from pathlib import Path
from collections import Counter

####VALIDATION IMAGE COUNT
# =========================
# EDIT THESE PATHS
# =========================
IMAGE_FOLDER = Path("/zpool/vladlab/data_drive/stimulus_sets/geogaze_open_images_stim/openimages_orig/images/valid")              # folder with image files (million+ ok)
IDS_TXT      = Path("/zpool/vladlab/active_drive/omaltz/scripts/geogaze/openimg_scripts/v1/filtered_image_index/validation_images_filtered.txt")
BBOX_CSV     = Path("/zpool/vladlab/active_drive/omaltz/scripts/geogaze/openimg_scripts/v1//filtered_bb_annotations/validation-annotations-bbox.filtered.csv")

IMG_EXTS = {".jpg", ".jpeg", ".png"}   # adjust if needed
IMAGE_ID_COL = "ImageID"
CHUNKSIZE = 500_000

# =========================
# 1) Count images in folder
# =========================
image_files = []
for ext in IMG_EXTS:
    image_files.extend(IMAGE_FOLDER.rglob(f"*{ext}"))

folder_image_ids = {
    p.stem for p in image_files
}

print(f"Images in folder: {len(folder_image_ids):,}")

# =========================
# 2) Count images in txt file
# =========================
with IDS_TXT.open("r") as f:
    txt_image_ids = {
        line.strip().split("/")[-1]
        for line in f
        if line.strip()
    }

print(f"Images listed in txt file: {len(txt_image_ids):,}")

# =========================
# 3) Count bounding boxes
# =========================
total_bboxes = 0
bbox_image_ids = Counter()

for df in pd.read_csv(BBOX_CSV, chunksize=CHUNKSIZE):
    total_bboxes += len(df)
    bbox_image_ids.update(df[IMAGE_ID_COL])

print(f"Total bounding boxes (rows): {total_bboxes:,}")
print(f"Unique ImageIDs with ≥1 bbox: {len(bbox_image_ids):,}")

# =========================
# 4) Bounding boxes with no associated image
#    (not in txt OR not in folder)
# =========================
invalid_bbox_rows = 0
invalid_bbox_image_ids = set()

for img_id, count in bbox_image_ids.items():
    if img_id not in txt_image_ids or img_id not in folder_image_ids:
        invalid_bbox_rows += count
        invalid_bbox_image_ids.add(img_id)

print(f"\nBounding boxes with no associated image:")
print(f"  Rows: {invalid_bbox_rows:,}")
print(f"  Unique ImageIDs: {len(invalid_bbox_image_ids):,}")

# =========================
# 5) Images with no bounding boxes
# =========================
images_with_no_bboxes = (
    txt_image_ids & folder_image_ids
) - set(bbox_image_ids.keys())

print(f"\nImages with NO bounding boxes:")
print(f"  Count: {len(images_with_no_bboxes):,}")

# Optional: preview first 20
print("\nExample ImageIDs with no bounding boxes:")
for img_id in list(images_with_no_bboxes)[:20]:
    print(f"  {img_id}")


Images in folder: 21,298
Images listed in txt file: 21,298
Total bounding boxes (rows): 45,182
Unique ImageIDs with ≥1 bbox: 21,298

Bounding boxes with no associated image:
  Rows: 0
  Unique ImageIDs: 0

Images with NO bounding boxes:
  Count: 0

Example ImageIDs with no bounding boxes:


In [2]:
### TEST, TRAIN, VALID IMAGE COUNT


CONFIG = {
    "train": {
        "bbox_csv": Path("/zpool/vladlab/active_drive/omaltz/scripts/geogaze/openimg_scripts/v1/filtered_bb_annotations/train-annotations-bbox.filtered.csv"),
        "ids_txt":  Path("/zpool/vladlab/active_drive/omaltz/scripts/geogaze/openimg_scripts/v1/filtered_image_index/train_images_filtered.txt"),
        "img_dir":  Path("/zpool/vladlab/data_drive/stimulus_sets/geogaze_open_images_stim/openimages_orig/images/train"),
    },
    "validation": {
        "bbox_csv": Path("//zpool/vladlab/active_drive/omaltz/scripts/geogaze/openimg_scripts/v1/filtered_bb_annotations/validation-annotations-bbox.filtered.csv"),
        "ids_txt":  Path("/zpool/vladlab/active_drive/omaltz/scripts/geogaze/openimg_scripts/v1/filtered_image_index/validation_images_filtered.txt"),
        "img_dir":  Path("/zpool/vladlab/data_drive/stimulus_sets/geogaze_open_images_stim/openimages_orig/images/valid"),
    },
    "test": {
        "bbox_csv": Path("/zpool/vladlab/active_drive/omaltz/scripts/geogaze/openimg_scripts/v1/filtered_bb_annotations/test-annotations-bbox.filtered.csv"),
        "ids_txt":  Path("/zpool/vladlab/active_drive/omaltz/scripts/geogaze/openimg_scripts/v1/filtered_image_index/test_images_filtered.txt"),
        "img_dir":  Path("/zpool/vladlab/data_drive/stimulus_sets/geogaze_open_images_stim/openimages_orig/images/test"),
    },
}

IMG_EXTS = {".jpg", ".jpeg", ".png"}  # adjust if needed
IMAGE_ID_COL = "ImageID"
CHUNKSIZE = 500_000

# =========================
# Helpers
# =========================
def load_txt_ids(txt_path: Path) -> set[str]:
    # lines like "train/0001eaf4..." -> keep the last path component
    with txt_path.open("r") as f:
        return {line.strip().split("/")[-1] for line in f if line.strip()}

def scan_folder_image_ids(img_dir: Path, exts: set[str]) -> set[str]:
    # Stem of filename (ImageID) for all images in folder (recursive)
    ids = set()
    for ext in exts:
        for p in img_dir.rglob(f"*{ext}"):
            ids.add(p.stem)
    return ids

def bbox_counts_from_csv(csv_path: Path, image_id_col: str, chunksize: int) -> tuple[int, Counter]:
    total_rows = 0
    per_image = Counter()
    for df in pd.read_csv(csv_path, chunksize=chunksize):
        if image_id_col not in df.columns:
            raise ValueError(f"{csv_path} missing column '{image_id_col}'. Columns: {list(df.columns)}")
        total_rows += len(df)
        per_image.update(df[image_id_col])
    return total_rows, per_image

# =========================
# Run per split
# =========================
results = {}

for split, cfg in CONFIG.items():
    bbox_csv = cfg["bbox_csv"]
    ids_txt  = cfg["ids_txt"]
    img_dir  = cfg["img_dir"]

    print(f"\n==================== {split.upper()} ====================")
    print(f"CSV : {bbox_csv}")
    print(f"TXT : {ids_txt}")
    print(f"DIR : {img_dir}")

    # 1) Counts for folder + txt
    folder_ids = scan_folder_image_ids(img_dir, IMG_EXTS)
    txt_ids = load_txt_ids(ids_txt)

    print(f"Images in folder (unique stems): {len(folder_ids):,}")
    print(f"Images in txt file (unique IDs): {len(txt_ids):,}")

    # 2) Bounding boxes
    total_bboxes, bbox_per_image = bbox_counts_from_csv(bbox_csv, IMAGE_ID_COL, CHUNKSIZE)
    print(f"Total bounding boxes (rows):     {total_bboxes:,}")
    print(f"Unique ImageIDs with ≥1 bbox:    {len(bbox_per_image):,}")

    # 3) BBoxes with no associated image in txt OR folder
    invalid_bbox_rows = 0
    invalid_bbox_image_ids = set()
    for img_id, c in bbox_per_image.items():
        if (img_id not in txt_ids) or (img_id not in folder_ids):
            invalid_bbox_rows += c
            invalid_bbox_image_ids.add(img_id)

    print(f"BBoxes missing associated image (not in txt OR not in folder):")
    print(f"  Rows:                          {invalid_bbox_rows:,}")
    print(f"  Unique ImageIDs:               {len(invalid_bbox_image_ids):,}")

    # 4) Images with no bounding boxes (restrict to images you “intended” and actually have)
    images_no_bboxes = (txt_ids & folder_ids) - set(bbox_per_image.keys())
    print(f"Images with NO bboxes (in txt ∩ folder, but not in CSV): {len(images_no_bboxes):,}")

    # store for summary + optional lists
    results[split] = {
        "folder_ids": folder_ids,
        "txt_ids": txt_ids,
        "total_bboxes": total_bboxes,
        "bbox_per_image": bbox_per_image,
        "invalid_bbox_rows": invalid_bbox_rows,
        "invalid_bbox_image_ids": invalid_bbox_image_ids,
        "images_no_bboxes": images_no_bboxes,
    }

    # show small samples
    if invalid_bbox_image_ids:
        print("  Example ImageIDs with invalid bboxes (first 10):")
        for x in list(invalid_bbox_image_ids)[:10]:
            print("   ", x)

    if images_no_bboxes:
        print("  Example ImageIDs with no bboxes (first 10):")
        for x in list(images_no_bboxes)[:10]:
            print("   ", x)

# =========================
# Compact summary table
# =========================
print("\n\n==================== SUMMARY ====================")
print(f"{'split':<12} {'folder_imgs':>12} {'txt_imgs':>12} {'bbox_rows':>12} {'bbox_bad_rows':>14} {'bbox_bad_imgs':>14} {'imgs_no_bbox':>14}")
for split in ["train", "validation", "test"]:
    r = results[split]
    print(
        f"{split:<12} "
        f"{len(r['folder_ids']):>12,} "
        f"{len(r['txt_ids']):>12,} "
        f"{r['total_bboxes']:>12,} "
        f"{r['invalid_bbox_rows']:>14,} "
        f"{len(r['invalid_bbox_image_ids']):>14,} "
        f"{len(r['images_no_bboxes']):>14,}"
    )



==================== TRAIN ====================
CSV : /zpool/vladlab/active_drive/omaltz/scripts/geogaze/openimg_scripts/v1/filtered_bb_annotations/train-annotations-bbox.filtered.csv
TXT : /zpool/vladlab/active_drive/omaltz/scripts/geogaze/openimg_scripts/v1/filtered_image_index/train_images_filtered.txt
DIR : /zpool/vladlab/data_drive/stimulus_sets/geogaze_open_images_stim/openimages_orig/images/train
Images in folder (unique stems): 1,064,384
Images in txt file (unique IDs): 1,064,384
Total bounding boxes (rows):     2,443,094
Unique ImageIDs with ≥1 bbox:    1,064,384
BBoxes missing associated image (not in txt OR not in folder):
  Rows:                          0
  Unique ImageIDs:               0
Images with NO bboxes (in txt ∩ folder, but not in CSV): 0

==================== VALIDATION ====================
CSV : //zpool/vladlab/active_drive/omaltz/scripts/geogaze/openimg_scripts/v1/filtered_bb_annotations/validation-annotations-bbox.filtered.csv
TXT : /zpool/vladlab/active_driv

In [3]:
import pandas as pd
from pathlib import Path
from collections import Counter
# =========================
# EDIT THESE PATHS
# =========================
BBOX_CSV   = Path("/zpool/vladlab/data_drive/geogaze_data/annotations/v5/filtered_bb_annotations/train-annotations-bbox.filtered.v5.csv")
IDS_TXT    = Path("/zpool/vladlab/data_drive/geogaze_data/annotations/v5/filtered_image_index/train_images_filtered.v5.txt")
LABELS_CSV = Path("/zpool/vladlab/active_drive/omaltz/scripts/geogaze/openimg_scripts/class-descriptions-boxable.csv")  # columns: LabelName,DisplayName (or LabelName,DisplayName)

IMAGE_ID_COL = "ImageID"
LABEL_COL    = "LabelName"
CHUNKSIZE    = 500_000

# =========================
# 1) Load mapping LabelName -> DisplayName
# =========================
labels_df = pd.read_csv(LABELS_CSV)

# Try to be robust to slightly different column names
# (some Open Images files use "DisplayName", some "DisplayName " etc.)
labels_df.columns = [c.strip() for c in labels_df.columns]

if "LabelName" not in labels_df.columns:
    raise ValueError(f"Mapping CSV must contain 'LabelName'. Columns found: {list(labels_df.columns)}")

# Pick a display/name column
display_col = None
for c in ["DisplayName", "display_name", "Name", "name"]:
    if c in labels_df.columns:
        display_col = c
        break
if display_col is None:
    # If your file is exactly two columns but named differently, just use the 2nd column
    if labels_df.shape[1] >= 2:
        display_col = labels_df.columns[1]
    else:
        raise ValueError(f"Couldn't find a display-name column. Columns found: {list(labels_df.columns)}")

label_to_name = dict(zip(labels_df["LabelName"], show := labels_df[display_col]))

print(f"Loaded label mapping rows: {len(label_to_name):,}")
print(f"Using display column: {display_col}")

# =========================
# 2) Load image IDs from txt
# =========================
with IDS_TXT.open("r") as f:
    valid_image_ids = {
        line.strip().split("/")[-1]
        for line in f
        if line.strip()
    }

print(f"Image IDs in txt file: {len(valid_image_ids):,}")

# =========================
# 3) Accumulate label counts per image (chunked)
# =========================
per_image_label_counts = {}  # img_id -> Counter(label -> count)

for df in pd.read_csv(BBOX_CSV, chunksize=CHUNKSIZE):
    df = df[df[IMAGE_ID_COL].isin(valid_image_ids)]
    if df.empty:
        continue

    for img_id, labels in df.groupby(IMAGE_ID_COL)[LABEL_COL]:
        if img_id not in per_image_label_counts:
            per_image_label_counts[img_id] = Counter()
        per_image_label_counts[img_id].update(labels)

print(f"Images with ≥1 bounding box: {len(per_image_label_counts):,}")

# =========================
# 4) Determine "top presence label" per image
#     (most frequent bbox label in that image)
# =========================
top_labels = []
for img_id, counter in per_image_label_counts.items():
    top_label, top_count = counter.most_common(1)[0]
    top_labels.append(top_label)

# =========================
# 5) Count images per top category
# =========================
category_counts = Counter(top_labels)

summary_df = (
    pd.DataFrame(
        [(lab, cnt) for lab, cnt in category_counts.items()],
        columns=["LabelName", "NumImages"]
    )
    .sort_values("NumImages", ascending=False)
    .reset_index(drop=True)
)

# Add human-readable names
summary_df["DisplayName"] = summary_df["LabelName"].map(label_to_name).fillna("(unknown label)")

# =========================
# 6) Add percentage of total images
# =========================
total_images = len(per_image_label_counts)

summary_df["PctImages"] = (
    summary_df["NumImages"] / total_images * 100
)

# Optional: nicer formatting
summary_df["PctImages"] = summary_df["PctImages"].round(2)


# Reorder columns for readability
summary_df = summary_df[["DisplayName", "LabelName", "NumImages", "PctImages"]]

print(summary_df.head(40))

Loaded label mapping rows: 601
Using display column: DisplayName
Image IDs in txt file: 112,279
Images with ≥1 bounding box: 112,279
         DisplayName   LabelName  NumImages  PctImages
0             Flower   /m/0c9ph5       7805       6.95
1               Boat    /m/019jd       4174       3.72
2                Dog   /m/0bt9lr       3979       3.54
3                Cat    /m/01yrx       3179       2.83
4               Bird    /m/015p6       2948       2.63
5                Boy   /m/01bl7v       2891       2.57
6              Table   /m/04bcr3       2448       2.18
7               Girl   /m/05r655       2169       1.93
8          Palm tree    /m/0cdl1       2161       1.92
9             Window    /m/0d4v4       2077       1.85
10             Chair   /m/01mzpv       1918       1.71
11            Poster   /m/01n5jq       1802       1.60
12         Sculpture    /m/06msq       1778       1.58
13             Train    /m/07jdr       1752       1.56
14               Bus    /m/01bjv       169

In [2]:
import pandas as pd
from collections import Counter

# =========================
# 1) Count number of boxes per image
# =========================
# image_id -> number of bounding boxes
boxes_per_image = {
    img_id: sum(counter.values())
    for img_id, counter in per_image_label_counts.items()
}

total_annotated_images = len(boxes_per_image)

print(f"Total images with ≥1 bounding box: {total_annotated_images:,}")

# =========================
# 2) Count images by number of boxes
# =========================
# num_boxes -> number of images
num_boxes_counter = Counter(boxes_per_image.values())

summary_boxes_df = (
    pd.DataFrame(
        [(num_boxes, num_images) for num_boxes, num_images in num_boxes_counter.items()],
        columns=["NumBoxes", "NumImages"]
    )
    .sort_values("NumBoxes")
    .reset_index(drop=True)
)

# =========================
# 3) Add percentages
# =========================
summary_boxes_df["PctImages"] = (
    summary_boxes_df["NumImages"] / total_annotated_images * 100
).round(2)

summary_boxes_df


Total images with ≥1 bounding box: 18,696


,NumBoxes,NumImages,PctImages
0,1,13959,74.66
1,2,3421,18.30
2,3,793,4.24
3,4,353,1.89
4,5,88,0.47
5,6,82,0.44


In [11]:
import pandas as pd
from pathlib import Path

# =========================
# EDIT THIS PATH
# =========================
BBOX_CSV = Path("/zpool/vladlab/data_drive/geogaze_data/annotations/v5/filtered_bb_annotations/train-annotations-bbox.filtered.v5.csv")
# =========================

# Load bounding box CSV
df = pd.read_csv(BBOX_CSV)

# Count unique bounding box categories
num_categories = df["LabelName"].nunique()

print(f"Number of unique bounding box categories: {num_categories}")

# (Optional) also print them
categories = sorted(df["LabelName"].unique())
print("\nBounding box categories:")
for c in categories:
    print(c)


Number of unique bounding box categories: 433

Bounding box categories:
/m/011k07
/m/012074
/m/0120dh
/m/01226z
/m/012n7d
/m/012w5l
/m/012xff
/m/012ysf
/m/0130jx
/m/013y1f
/m/01432t
/m/014y4n
/m/0152hh
/m/015p6
/m/015qbp
/m/015qff
/m/0162_1
/m/0167gd
/m/016m2d
/m/0174k2
/m/0174n1
/m/0175cv
/m/0176mf
/m/017ftj
/m/018j2
/m/018p4k
/m/01940j
/m/0199g
/m/019h78
/m/019jd
/m/019w40
/m/01_5g
/m/01b638
/m/01b7fy
/m/01bfm9
/m/01bjv
/m/01bl7v
/m/01bms0
/m/01bqk0
/m/01btn
/m/01c648
/m/01cmb2
/m/01d380
/m/01dws
/m/01dxs
/m/01dy8n
/m/01f8m5
/m/01fh4r
/m/01g3x7
/m/01gkx_
/m/01gllr
/m/01gmv2
/m/01h3n
/m/01h44
/m/01h8tj
/m/01j3zr
/m/01j4z9
/m/01j51
/m/01j5ks
/m/01j61q
/m/01jfm_
/m/01jfsr
/m/01k6s3
/m/01kb5b
/m/01knjb
/m/01krhy
/m/01lcw4
/m/01llwg
/m/01lsmm
/m/01lynh
/m/01m2v
/m/01m4t
/m/01mqdt
/m/01mzpv
/m/01n4qj
/m/01n5jq
/m/01nq26
/m/01pns0
/m/01r546
/m/01rkbr
/m/01s55n
/m/01vbnl
/m/01x3jk
/m/01x3z
/m/01x_v
/m/01xq0k1
/m/01xqw
/m/01xs3r
/m/01xygc
/m/01y9k5
/m/01yrx
/m/01yx86
/m/02068x
/m/020jm
/m/020